In [ ]:
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

with open('manifestacoes.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

# Filtrar textos longos (> 500 caracteres)
textos_longos = [d for d in dados if len(d['texto']) > 500][:5]

# Configuração 1 (Baixo overlap) e Configuração 2 (Overlap ideal)
splitter_1 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
splitter_2 = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=80)

chunks_totais = []
cores = ['red', 'blue', 'green', 'purple', 'orange']
labels = []

# Gerando chunks com a Configuração 2
for idx, doc in enumerate(textos_longos):
    chunks = splitter_2.split_text(doc['texto'])
    chunks_totais.extend(chunks)
    labels.extend([cores[idx]] * len(chunks))

modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
emb_chunks = modelo.encode(chunks_totais)

# PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(emb_chunks)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=labels, s=100, alpha=0.7)
plt.title("Visualização PCA dos Chunks")
plt.show()

# Respostas para o Notebook:
# - A sobreposição (overlap) maior garante que dependências contextuais (sujeito e problema relatado) 
#   não sejam perdidas na divisão brusca do texto.
# - A Configuração 2 (chunk_size=200, chunk_overlap=80) preserva o sentido das denúncias ao manter conexões 
#   como "acúmulo de lixo" e "mau cheiro insuportável".
# - A visualização em 2D indica que os chunks de uma mesma manifestação se aproximam espacialmente, validando 
#   a coesão do espaço vetorial.